In [0]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# =========================
# Paths
# =========================
save_path = "/Volumes/workspace/default/ather/"
fig_path = "/Volumes/workspace/default/ather/figures/"
os.makedirs(fig_path, exist_ok=True)

In [0]:
# =========================
# Load Data
# =========================
df = spark.table('workspace.ather.ather_table').toPandas()

In [0]:
# =========================
# Define Stress Indicators
# =========================
stress_indicators = [
    "Motor_current_Mean",
    "Motor_current_Max",
    "Motor_Torque_Mean",
    "Motor_Torque_Max",
    "Motor_speed_Mean",
    "Motor_speed_Max",
    "Thermal_Load_Index"
]

# Keep only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

In [0]:
# =========================
# Compute Stress Influence Score
# =========================
scores = []

for col in numeric_df.columns:
    if col in stress_indicators:
        continue
    
    corr_values = []
    
    for stress_col in stress_indicators:
        if stress_col in numeric_df.columns:
            corr = numeric_df[col].corr(numeric_df[stress_col])
            if pd.notnull(corr):
                corr_values.append(abs(corr))
    
    if len(corr_values) > 0:
        score = np.mean(corr_values)
    else:
        score = 0
    
    scores.append((col, score))


In [0]:
# =========================
# Create Ranking Table
# =========================
ranking_df = pd.DataFrame(scores, columns=["Parameter", "Stress_Influence_Score"])
ranking_df = ranking_df.sort_values(by="Stress_Influence_Score", ascending=False).reset_index(drop=True)


In [0]:
# =========================
# Save Ranking Table
# =========================
ranking_df.to_csv(save_path + "stress_parameter_ranking.csv", index=False)

In [0]:
# =========================
# Plot Top-N Parameters
# =========================
TOP_N = 20
top_df = ranking_df.head(TOP_N)

plt.figure(figsize=(10, 8))
sns.barplot(x="Stress_Influence_Score", y="Parameter", data=top_df)

plt.title(f"Top {TOP_N} Stress Influencing Parameters")
plt.tight_layout()

plt.savefig(fig_path + "top_stress_parameters.png", dpi=300)
plt.show()

In [0]:
# =========================
# Categorize Parameters
# =========================
def categorize(param):
    p = param.lower()
    
    if "current" in p:
        return "Motor Current"
    elif "torque" in p:
        return "Torque"
    elif "speed" in p:
        return "Speed"
    elif "temp" in p or "thermal" in p:
        return "Thermal"
    elif any(k in p for k in ["soc", "soh", "soe", "cell", "voltage"]):
        return "Battery Signals"
    else:
        return "Other"

ranking_df["Category"] = ranking_df["Parameter"].apply(categorize)


In [0]:
# =========================
# Top Influencing Categories
# =========================
category_scores = ranking_df.groupby("Category")["Stress_Influence_Score"].mean().sort_values(ascending=False)

print("Top Influencing Categories:")
print(category_scores.head(5))